# keigo-slm — QLoRA fine-tuning (Colab)

Trains Qwen3 to preserve Japanese politeness register in JP→EN translation, using the
verified seed data in the repo (333 rows, balanced across informal/polite/formal).

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough for 0.6B–1.7B).
Then add secrets via the 🔑 panel: `HF_TOKEN` (required), `TEACHER_API_KEY` (optional, for eval).

In [ ]:
# 1. Install
!pip install -q unsloth "trl<0.10" peft datasets

In [ ]:
# 2. Get the code + log in to Hugging Face
!git clone -q https://github.com/akabraham06/keigo-slm.git
%cd keigo-slm
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import login; login(os.environ['HF_TOKEN'])

In [ ]:
# 3. (optional) regenerate the compositional seed — already committed, so usually skip
# !PYTHONPATH=src python -m data.build_seed
!wc -l data/seed/*.jsonl

In [ ]:
# 4. Train (QLoRA SFT). Uses the versioned code in src/train/sft.py.
#    Push to your Hub repo by adding --push-repo; drop it to train locally only.
!PYTHONPATH=src python -m train.sft --model unsloth/Qwen3-1.7B --epochs 3 \
    --push-repo akabraham06/keigo-slm-qwen3-1.7b

In [ ]:
# 5. Sanity check: does the tuned model track register across a contrastive triple?
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained('outputs/keigo-sft', max_seq_length=1024, load_in_4bit=True)
FastLanguageModel.for_inference(model)
tmpl = open('prompts/slm_inference.txt').read()
for jp in ['資料見とくね。', '資料を見ておきます。', '資料を拝見いたします。']:
    ids = tok(tmpl.replace('{JP}', jp), return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=64, do_sample=False)
    print(jp, '->', tok.decode(out[0], skip_special_tokens=True).split('English:')[-1].strip())

## Next

- The push step publishes your adapter (deliverable #2).
- Run the eval (`python -m eval.run_eval`, needs a teacher/judge key) to get the
  base-vs-tuned-vs-LLM table on the 127-row golden set.
- 333 rows is a small first run — expect movement but not a final number. Scale with
  `data.generate` (teacher) for phrasing diversity, then retrain.